In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt


In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors

X_train_tensor = torch.FloatTensor(X_train)
X_test_tensor = torch.FloatTensor(X_test)
y_train_tensor = torch.FloatTensor(y_train).reshape(-1, 1)
y_test_tensor = torch.FloatTensor(y_test).reshape(-1, 1)

print("\n=== Tensors Created ===")
print(f"X_train_tensor shape: {X_train_tensor.shape}")
print(f"X_test_tensor shape: {X_test_tensor.shape}")
print(f"y_train_tensor shape: {y_train_tensor.shape}")
print(f"y_test_tensor shape: {y_test_tensor.shape}")


In [ ]:
# 2. Create TensorDataset objects

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

print(f"\nTrain dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")


In [ ]:
# 3. Create DataLoaders

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"\nNumber of training batches: {len(train_loader)}")
print(f"Number of testing batches: {len(test_loader)}")


In [ ]:
# 4. Print shape of one batch
for X_batch, y_batch in train_loader:
    print(f"\n=== One Batch Inspection ===")
    print(f"X_batch shape: {X_batch.shape}")
    print(f"y_batch shape: {y_batch.shape}")
    break


In [ ]:
# 5. Display sample images

fig, axes = plt.subplots(2, 4, figsize=(15, 8))
axes = axes.flatten()

for i in range(8):
    # Get image and age
    img = X_train[i].transpose(1, 2, 0)  # Convert from (C, H, W) to (H, W, C)
    age = y_train[i]

    # Display
    axes[i].imshow(img)
    axes[i].set_title(f'Age: {age:.0f}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Task 1: Write your model class here:
class AgePredictor(nn.Module):
    def __init__(self, input_size):
        super(AgePredictor, self).__init__()

        # Flatten layer
        self.flatten = nn.Flatten()

        # 4 Linear layers with ReLU activation
        self.fc1 = nn.Linear(input_size, 512)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.3)

        self.fc2 = nn.Linear(512, 256)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)

        self.fc3 = nn.Linear(256, 128)
        self.relu3 = nn.ReLU()
        self.dropout3 = nn.Dropout(0.3)

        self.fc4 = nn.Linear(128, 1)  # Output layer for age prediction

    def forward(self, x):
        x = self.flatten(x)

        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)

        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)

        x = self.fc3(x)
        x = self.relu3(x)
        x = self.dropout3(x)

        x = self.fc4(x)

        return x

In [ ]:
# Task 2: Write your training loop here:
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        # Move data to device
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    return avg_loss

In [ ]:
# Task 3: Write your validation loop here:
def validate_epoch(model, test_loader, criterion, device):
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            # Move data to device
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            # Forward pass
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            running_loss += loss.item()

    avg_loss = running_loss / len(test_loader)
    return avg_loss

In [ ]:
# Task 4: Define device, model, loss, optimizer:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n=== Training Setup ===")
print(f"Using device: {device}")

# Calculate input size (C * H * W)
input_size = X_train.shape[1] * X_train.shape[2] * X_train.shape[3]
print(f"Input size: {input_size}")

# Initialize model
model = AgePredictor(input_size).to(device)
print(f"\nModel architecture:")
print(model)

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Task 5: Start training for 20 epochs:
num_epochs = 20
train_losses = []
val_losses = []

print(f"\n=== Training Started ===")
for epoch in range(num_epochs):
    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(train_loss)

    # Validate
    val_loss = validate_epoch(model, test_loader, criterion, device)
    val_losses.append(val_loss)

    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

print("\n=== Training Completed ==")

In [ ]:
# Task 1: Write your code here:
plt.figure(figsize=(10, 6))
plt.plot(range(1, num_epochs + 1), train_losses, label='Training Loss', marker='o')
plt.plot(range(1, num_epochs + 1), val_losses, label='Validation Loss', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Training and Validation Loss Over Epochs')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here:
model.eval()
with torch.no_grad():
    # Get a batch from test set
    X_sample = X_test_tensor[:8].to(device)
    y_actual = y_test[:8]

    # Predict
    predictions = model(X_sample).cpu().numpy().flatten()

# Display images with predictions
fig, axes = plt.subplots(2, 4, figsize=(15, 8))
axes = axes.flatten()

for i in range(8):
    # Get image
    img = X_test[i].transpose(1, 2, 0)  # Convert from (C, H, W) to (H, W, C)
    actual_age = y_actual[i]
    predicted_age = predictions[i]

    # Display
    axes[i].imshow(img)
    axes[i].set_title(f'Actual: {actual_age:.0f}\nPredicted: {predicted_age:.0f}')
    axes[i].axis('off')

plt.suptitle('Age Prediction Results (Actual vs Predicted)', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# Calculate MAE for test set
model.eval()
with torch.no_grad():
    all_predictions = model(X_test_tensor.to(device)).cpu().numpy().flatten()
    mae = np.mean(np.abs(all_predictions - y_test))
    print(f"\n=== Final Evaluation ===")
    print(f"Mean Absolute Error (MAE) on test set: {mae:.2f} years")